<a href="https://colab.research.google.com/github/geopayme/AstroPhysics/blob/main/BeatLab_AI_Artifact_Remover.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎛️ BeatLab — AI Audio Artifact Detector & Remover
> **Google Colab Tool** — Companion to the Beat Lab web app  
> Upload a WAV or MP3, detect AI-generation signatures, and export a processed version with artifacts reduced.

---
### What this tool does
| Stage | What happens |
|---|---|
| **1. Analysis** | Measures 6 signal characteristics known to differ between AI-generated and human-produced audio |
| **2. Scoring** | Produces a suspicion score (0–100%) with per-metric breakdown |
| **3. Processing** | Applies targeted DSP to reduce the flagged artifacts |
| **4. Export** | Downloads the processed audio as a 44.1 kHz 24-bit WAV |

### Limitations
- This is **heuristic** analysis, not a neural classifier. It can produce false positives on heavily mastered or heavily quantised human productions.
- Processing improves naturalness; it cannot "un-generate" AI content.
- For professional use, treat the output as a starting point for further mixing.


In [1]:
# ── Cell 1: Install dependencies ─────────────────────────────────────
!pip install -q librosa soundfile noisereduce matplotlib numpy scipy

In [2]:
# ── Cell 2: Imports ───────────────────────────────────────────────────
import numpy as np
import librosa
import librosa.display
import soundfile as sf
import noisereduce as nr
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import signal as scipy_signal
from scipy.stats import variation
from google.colab import files
import io, os, warnings
warnings.filterwarnings('ignore')

print('✅ All dependencies loaded')

✅ All dependencies loaded


In [ ]:
# ── Cell 3: Upload your audio file ───────────────────────────────────
print('📂 Select a WAV or MP3 file to analyse...')
uploaded = files.upload()
fname = list(uploaded.keys())[0]
audio_bytes = uploaded[fname]

# Load with librosa (handles WAV, MP3, FLAC, OGG)
y, sr = librosa.load(io.BytesIO(audio_bytes), sr=None, mono=False)

# Ensure stereo
if y.ndim == 1:
    y = np.stack([y, y])

y_mono = librosa.to_mono(y)
duration = len(y_mono) / sr

print(f'\n✅ Loaded: {fname}')
print(f'   Sample rate : {sr} Hz')
print(f'   Duration    : {duration:.2f} s')
print(f'   Channels    : {y.shape[0]}')
print(f'   Samples     : {y.shape[1]:,}')

In [ ]:
# ── Cell 4: AI Artifact Analysis ─────────────────────────────────────

results = {}

# ── Metric 1: Dynamic Range (crest factor) ───────────────────────────
peak  = np.max(np.abs(y_mono))
rms   = np.sqrt(np.mean(y_mono ** 2))
crest_db = 20 * np.log10(peak / rms) if rms > 0 else 20
dr_score = max(0, min(1, (16 - crest_db) / 9))

results['Dynamic Range'] = {
    'value'  : f'{crest_db:.1f} dB',
    'score'  : dr_score,
    'detail' : 'Over-limited (AI hallmark)' if crest_db < 8
               else 'Compressed' if crest_db < 12 else 'Natural',
    'raw'    : crest_db
}

# ── Metric 2: Spectral Flatness ───────────────────────────────────────
flatness = librosa.feature.spectral_flatness(y=y_mono)
mean_flat = float(np.mean(flatness))
sf_score  = (max(0, min(1, (mean_flat - 0.25) / 0.3)) if mean_flat > 0.25
             else max(0, min(1, (0.03 - mean_flat) / 0.025)) if mean_flat < 0.03
             else 0)

results['Spectral Flatness'] = {
    'value'  : f'{mean_flat:.4f}',
    'score'  : sf_score,
    'detail' : 'Unnaturally flat' if mean_flat > 0.25
               else 'Over-processed (peaky)' if mean_flat < 0.03 else 'Normal',
    'raw'    : mean_flat
}

# ── Metric 3: Timing Regularity (beat jitter) ────────────────────────
tempo, beat_frames = librosa.beat.beat_track(y=y_mono, sr=sr)
beat_times = librosa.frames_to_time(beat_frames, sr=sr)
if len(beat_times) > 4:
    ibi       = np.diff(beat_times) * 1000  # ms
    jitter_ms = float(np.std(ibi))
    tr_score  = max(0, min(1, (8 - jitter_ms) / 7))
else:
    jitter_ms = 999
    tr_score  = 0.5

results['Timing Regularity'] = {
    'value'  : f'{jitter_ms:.1f} ms jitter' if jitter_ms < 900 else 'N/A',
    'score'  : tr_score,
    'detail' : 'Machine-perfect' if jitter_ms < 3
               else 'Suspiciously tight' if jitter_ms < 8 else 'Natural jitter',
    'raw'    : jitter_ms
}

# ── Metric 4: Stereo Width (mid-side ratio) ──────────────────────────
mid  = (y[0] + y[1]) * 0.5
side = (y[0] - y[1]) * 0.5
mid_energy  = np.mean(mid  ** 2)
side_energy = np.mean(side ** 2)
ms_ratio = side_energy / mid_energy if mid_energy > 0 else 0
sw_score = (max(0, min(1, (ms_ratio - 0.75) / 0.4)) if ms_ratio > 0.75
            else max(0, min(1, (0.03 - ms_ratio) / 0.025)) if ms_ratio < 0.03
            else 0)

results['Stereo Width'] = {
    'value'  : f'{ms_ratio:.3f} M/S ratio',
    'score'  : sw_score,
    'detail' : 'Unnaturally wide' if ms_ratio > 0.75
               else 'Near-mono' if ms_ratio < 0.03 else 'Balanced',
    'raw'    : ms_ratio
}

# ── Metric 5: High-Frequency Energy Ratio ────────────────────────────
S = np.abs(librosa.stft(y_mono))
freqs = librosa.fft_frequencies(sr=sr)
hf_mask   = freqs > 15000
hf_energy = np.mean(S[hf_mask, :] ** 2)
tot_energy= np.mean(S ** 2)
hf_ratio  = float(hf_energy / tot_energy) if tot_energy > 0 else 0
hf_score  = (max(0, min(1, (0.002 - hf_ratio) / 0.002)) if hf_ratio < 0.002
             else max(0, min(1, (hf_ratio - 0.07) / 0.06)) if hf_ratio > 0.07
             else 0)

results['HF Energy'] = {
    'value'  : f'{hf_ratio*100:.3f}%',
    'score'  : hf_score,
    'detail' : 'HF dead (AI cutoff)' if hf_ratio < 0.002
               else 'Artificially boosted' if hf_ratio > 0.07 else 'Normal',
    'raw'    : hf_ratio
}

# ── Metric 6: Noise Floor Smoothness ────────────────────────────────
# AI generators have an unusually smooth/absent noise floor
# We measure the quietest 5% of RMS frames
frame_sz = 2048
rms_frames = librosa.feature.rms(y=y_mono, frame_length=frame_sz, hop_length=frame_sz//2)[0]
quiet_rms  = np.percentile(rms_frames, 5)
db_floor   = 20 * np.log10(quiet_rms + 1e-10)
# Typical mastered music: floor around -60 to -80 dB
# AI music often has a suspiciously clean floor below -90 dB
nf_score   = max(0, min(1, (-db_floor - 80) / 30))

results['Noise Floor'] = {
    'value'  : f'{db_floor:.1f} dBFS',
    'score'  : nf_score,
    'detail' : 'Suspiciously clean (AI)' if db_floor < -90
               else 'Very quiet' if db_floor < -75 else 'Normal',
    'raw'    : db_floor
}

# ── Combined Score ────────────────────────────────────────────────────
weights = {
    'Dynamic Range'    : 0.25,
    'Spectral Flatness': 0.18,
    'Timing Regularity': 0.22,
    'Stereo Width'     : 0.10,
    'HF Energy'        : 0.12,
    'Noise Floor'      : 0.13,
}
combined = sum(results[k]['score'] * w for k, w in weights.items())
verdict  = ('🔴 LIKELY AI-GENERATED' if combined > 0.65
            else '🟡 UNCERTAIN / AMBIGUOUS' if combined > 0.38
            else '🟢 LIKELY HUMAN-PRODUCED')

print('\n' + '═'*55)
print(f'  AI ARTIFACT ANALYSIS — {fname}')
print('═'*55)
for name, r in results.items():
    bar = '█' * int(r['score'] * 20) + '░' * (20 - int(r['score'] * 20))
    print(f'  {name:<22} {bar}  {r["value"]:>15}  {r["detail"]}')
print('─'*55)
print(f'  SUSPICION SCORE    {combined*100:>5.1f}%')
print(f'  VERDICT            {verdict}')
print('═'*55)

In [ ]:
# ── Cell 5: Visualisation Dashboard ──────────────────────────────────
fig = plt.figure(figsize=(16, 12), facecolor='#07070c')
fig.suptitle(f'Beat Lab — AI Signal Analysis\n{fname}',
             color='#f59e0b', fontsize=14, fontweight='bold', y=0.98)

gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)

ax_style = dict(facecolor='#111118', labelcolor='#9090c0',
                titlecolor='#f0eff8', spines_color='#22223a')

def style_ax(ax, title):
    ax.set_facecolor('#111118')
    ax.set_title(title, color='#f0eff8', fontsize=10, pad=8)
    for spine in ax.spines.values(): spine.set_color('#22223a')
    ax.tick_params(colors='#6868a0', labelsize=8)
    ax.xaxis.label.set_color('#6868a0')
    ax.yaxis.label.set_color('#6868a0')

# Plot 1: Waveform
ax1 = fig.add_subplot(gs[0, :])
times = np.linspace(0, duration, len(y_mono))
ax1.plot(times, y_mono, color='#f59e0b', linewidth=0.4, alpha=0.85)
if len(beat_times) > 0:
    for bt in beat_times:
        ax1.axvline(bt, color='#f87171', alpha=0.5, linewidth=0.8)
ax1.set_xlabel('Time (s)'); ax1.set_ylabel('Amplitude')
style_ax(ax1, 'Waveform + Beat Markers')

# Plot 2: Spectrogram
ax2 = fig.add_subplot(gs[1, 0])
D = librosa.amplitude_to_db(np.abs(librosa.stft(y_mono)), ref=np.max)
librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='hz',
                         ax=ax2, cmap='magma')
ax2.set_ylim(0, min(22050, sr//2))
style_ax(ax2, 'Spectrogram')

# Plot 3: Spectral Flatness over time
ax3 = fig.add_subplot(gs[1, 1])
flat_times = librosa.frames_to_time(np.arange(len(flatness[0])), sr=sr)
ax3.plot(flat_times, flatness[0], color='#7dd3fc', linewidth=0.7)
ax3.axhline(0.25, color='#f87171', linestyle='--', linewidth=1, label='AI threshold')
ax3.set_xlabel('Time (s)'); ax3.set_ylabel('Flatness')
ax3.legend(fontsize=7, labelcolor='#9090c0', facecolor='#111118', edgecolor='#22223a')
style_ax(ax3, 'Spectral Flatness (high = suspicious)')

# Plot 4: Metric Scores Bar Chart
ax4 = fig.add_subplot(gs[2, 0])
names  = list(results.keys())
scores = [results[n]['score'] for n in names]
colors = ['#f87171' if s > 0.65 else '#f59e0b' if s > 0.38 else '#4ade80' for s in scores]
bars = ax4.barh(names, scores, color=colors, height=0.55)
ax4.axvline(0.65, color='#f87171', linestyle=':', linewidth=1, alpha=0.7)
ax4.axvline(0.38, color='#f59e0b', linestyle=':', linewidth=1, alpha=0.7)
ax4.set_xlim(0, 1); ax4.set_xlabel('Suspicion Score')
for bar, s in zip(bars, scores):
    ax4.text(s + 0.02, bar.get_y() + bar.get_height()/2,
             f'{s*100:.0f}%', va='center', ha='left', color='#f0eff8', fontsize=8)
style_ax(ax4, 'Per-Metric Suspicion Scores')

# Plot 5: Combined verdict gauge
ax5 = fig.add_subplot(gs[2, 1])
theta = np.linspace(np.pi, 0, 300)
ax5.plot(np.cos(theta), np.sin(theta), color='#22223a', linewidth=12, solid_capstyle='round')
theta_score = np.linspace(np.pi, np.pi - combined * np.pi, 300)
gauge_color = '#f87171' if combined > 0.65 else '#f59e0b' if combined > 0.38 else '#4ade80'
ax5.plot(np.cos(theta_score), np.sin(theta_score),
         color=gauge_color, linewidth=12, solid_capstyle='round')
ax5.text(0, 0.15, f'{combined*100:.0f}%', ha='center', va='center',
         color=gauge_color, fontsize=28, fontweight='bold')
verdict_short = 'LIKELY AI' if combined > 0.65 else 'UNCERTAIN' if combined > 0.38 else 'LIKELY HUMAN'
ax5.text(0, -0.25, verdict_short, ha='center', va='center',
         color=gauge_color, fontsize=11, fontweight='bold')
ax5.set_xlim(-1.3, 1.3); ax5.set_ylim(-0.5, 1.2)
ax5.axis('off')
ax5.set_facecolor('#111118')
ax5.set_title('Combined Suspicion Score', color='#f0eff8', fontsize=10, pad=8)

plt.savefig('beatlab_ai_analysis.png', dpi=150, bbox_inches='tight',
            facecolor='#07070c', edgecolor='none')
plt.show()
print('📊 Dashboard saved as beatlab_ai_analysis.png')

In [ ]:
# ── Cell 6: Processing Configuration ─────────────────────────────────
# Adjust these sliders to control how aggressively each artifact is treated.
# Set to 0.0 to skip a processing stage.

CFG = {
    # Dynamic range expansion — restores punch lost to over-limiting
    # 0 = off, 1 = subtle (recommended), 2 = aggressive
    'dr_expansion'       : 1.0,

    # Spectral smoothing — reduces unnatural frequency spikes
    # 0 = off, 1 = light, 3 = heavy
    'spectral_smooth'    : 1.0,

    # Humanisation — adds subtle micro-timing jitter to beat events
    # 0 = off, 1 = subtle (±4 ms), 2 = obvious (±12 ms)
    'humanise_timing'    : 0.8,

    # Stereo correction — narrows an unnaturally wide stereo field
    # 0 = off, 1 = full correction toward natural target (0.5 M/S ratio)
    'stereo_correct'     : 0.7,

    # HF restoration — gentle high-shelf boost if HF energy was detected as dead
    # 0 = off, amount in dB (1–4 recommended)
    'hf_restore_db'      : 2.0,

    # Noise floor injection — adds very low-level pink noise to restore
    # the organic noise floor that AI audio lacks
    # 0 = off, level in dBFS below full scale (-90 recommended)
    'noise_floor_db'     : -90.0,

    # Output level — final normalisation target in dBFS (-1 = standard master)
    'output_lufs_target' : -1.0,
}

print('✅ Configuration loaded:')
for k, v in CFG.items():
    print(f'   {k:<24} = {v}')

In [ ]:
# ── Cell 7: Processing Pipeline ───────────────────────────────────────
import copy

y_proc = y.copy().astype(np.float32)
log = []

# ── Stage 1: Dynamic Range Expansion ─────────────────────────────────
if CFG['dr_expansion'] > 0 and results['Dynamic Range']['raw'] < 14:
    # Upward expansion: boost samples near silence, leave peaks alone
    # This restores the "space" between transients that limiting removed
    strength = CFG['dr_expansion']
    rms_proc = np.sqrt(np.mean(y_proc[0] ** 2))
    threshold = rms_proc * 0.35
    for ch in range(y_proc.shape[0]):
        mask = np.abs(y_proc[ch]) < threshold
        # Gentle downward scaling of sub-threshold samples (expansion)
        scale_factor = 1 - (strength * 0.12 * (1 - np.abs(y_proc[ch][mask]) / threshold))
        y_proc[ch][mask] *= scale_factor
    log.append(f'✓ Dynamic range expansion (strength={CFG["dr_expansion"]})')
else:
    log.append('  Dynamic range expansion — skipped (DR already natural)')

# ── Stage 2: Spectral Smoothing ───────────────────────────────────────
if CFG['spectral_smooth'] > 0 and (results['Spectral Flatness']['score'] > 0.25 or
                                    results['HF Energy']['score'] > 0.3):
    strength = CFG['spectral_smooth']
    for ch in range(y_proc.shape[0]):
        # Apply noisereduce with a mild setting — it targets spectral anomalies
        # Use the quietest portion as the "noise" profile
        noise_clip = y_proc[ch][:sr]  # first second as reference
        y_proc[ch] = nr.reduce_noise(
            y=y_proc[ch], y_noise=noise_clip, sr=sr,
            prop_decrease=0.25 * strength,
            freq_mask_smooth_hz=500,
            time_mask_smooth_ms=100
        ).astype(np.float32)
    log.append(f'✓ Spectral smoothing (strength={CFG["spectral_smooth"]})')
else:
    log.append('  Spectral smoothing — skipped')

# ── Stage 3: Stereo Width Correction ─────────────────────────────────
if CFG['stereo_correct'] > 0 and results['Stereo Width']['score'] > 0.2:
    ms_ratio_now = results['Stereo Width']['raw']
    target_ratio = 0.50  # natural-sounding M/S ratio
    if ms_ratio_now > target_ratio:
        # Too wide: attenuate the side channel
        side_reduction = CFG['stereo_correct'] * min(1, (ms_ratio_now - target_ratio) / 0.5)
        mid_ch  = (y_proc[0] + y_proc[1]) * 0.5
        side_ch = (y_proc[0] - y_proc[1]) * 0.5 * (1 - side_reduction * 0.4)
        y_proc[0] = mid_ch + side_ch
        y_proc[1] = mid_ch - side_ch
        log.append(f'✓ Stereo width reduced ({ms_ratio_now:.3f} → ~{target_ratio:.2f} M/S ratio)')
    else:
        log.append('  Stereo correction — skipped (already narrow enough)')
else:
    log.append('  Stereo correction — skipped')

# ── Stage 4: HF Restoration ──────────────────────────────────────────
if CFG['hf_restore_db'] > 0 and results['HF Energy']['raw'] < 0.003:
    # High-shelf boost above 12 kHz using a simple FIR
    gain_lin = 10 ** (CFG['hf_restore_db'] / 20)
    nyq = sr / 2
    cutoff = min(12000 / nyq, 0.99)
    b_hf, a_hf = scipy_signal.butter(2, cutoff, btype='high')
    for ch in range(y_proc.shape[0]):
        hf_component = scipy_signal.filtfilt(b_hf, a_hf, y_proc[ch]).astype(np.float32)
        y_proc[ch] = y_proc[ch] + hf_component * (gain_lin - 1)
    log.append(f'✓ HF restoration +{CFG["hf_restore_db"]} dB shelf above 12 kHz')
else:
    log.append('  HF restoration — skipped')

# ── Stage 5: Noise Floor Injection ───────────────────────────────────
if CFG['noise_floor_db'] < -60 and results['Noise Floor']['score'] > 0.3:
    noise_level = 10 ** (CFG['noise_floor_db'] / 20)
    # Pink noise approximation: filter white noise with 1/f rolloff
    white = np.random.randn(y_proc.shape[1]).astype(np.float32)
    b_pink = np.array([0.049922035, -0.095993537, 0.050612699, -0.004408786])
    a_pink = np.array([1, -2.494956002, 2.017265875, -0.522189400])
    pink = scipy_signal.lfilter(b_pink, a_pink, white).astype(np.float32)
    pink = pink / (np.max(np.abs(pink)) + 1e-10) * noise_level
    for ch in range(y_proc.shape[0]):
        y_proc[ch] += pink
    log.append(f'✓ Noise floor injection at {CFG["noise_floor_db"]} dBFS')
else:
    log.append('  Noise floor injection — skipped')

# ── Stage 6: Output Normalisation ────────────────────────────────────
target_peak = 10 ** (CFG['output_lufs_target'] / 20)
current_peak = np.max(np.abs(y_proc))
if current_peak > 0:
    y_proc = y_proc * (target_peak / current_peak)
log.append(f'✓ Normalised to {CFG["output_lufs_target"]} dBFS peak')

print('\n🎛️  Processing complete:')
for l in log: print(f'   {l}')

In [ ]:
# ── Cell 8: Before / After Comparison ────────────────────────────────
y_orig_mono = librosa.to_mono(y)
y_proc_mono = librosa.to_mono(y_proc)

fig, axes = plt.subplots(2, 2, figsize=(16, 8), facecolor='#07070c')
fig.suptitle('Before vs After Processing', color='#f59e0b',
             fontsize=13, fontweight='bold')

titles = [('Original Waveform', '#f59e0b'),
          ('Processed Waveform', '#4ade80'),
          ('Original Spectrogram', '#f59e0b'),
          ('Processed Spectrogram', '#4ade80')]

for ax, (title, col) in zip(axes.flat, titles):
    ax.set_facecolor('#111118')
    for sp in ax.spines.values(): sp.set_color('#22223a')
    ax.tick_params(colors='#6868a0', labelsize=8)
    ax.set_title(title, color=col, fontsize=10)

t_axis = np.linspace(0, duration, len(y_orig_mono))
axes[0,0].plot(t_axis, y_orig_mono, color='#f59e0b', lw=0.3, alpha=0.9)
axes[0,1].plot(t_axis, y_proc_mono, color='#4ade80', lw=0.3, alpha=0.9)

D_orig = librosa.amplitude_to_db(np.abs(librosa.stft(y_orig_mono)), ref=np.max)
D_proc = librosa.amplitude_to_db(np.abs(librosa.stft(y_proc_mono)), ref=np.max)
librosa.display.specshow(D_orig, sr=sr, x_axis='time', y_axis='hz',
                         ax=axes[1,0], cmap='magma')
librosa.display.specshow(D_proc, sr=sr, x_axis='time', y_axis='hz',
                         ax=axes[1,1], cmap='viridis')

plt.tight_layout()
plt.savefig('beatlab_before_after.png', dpi=150, bbox_inches='tight',
            facecolor='#07070c')
plt.show()
print('📊 Comparison saved as beatlab_before_after.png')

In [ ]:
# ── Cell 9: Export Processed Audio ───────────────────────────────────
stem = os.path.splitext(fname)[0]
out_name = f'{stem}_processed.wav'

# Write as 24-bit WAV, stereo, at original sample rate
sf.write(out_name, y_proc.T, sr, subtype='PCM_24')

print(f'✅ Exported: {out_name}')
print(f'   Format    : WAV 24-bit PCM')
print(f'   Rate      : {sr} Hz')
print(f'   Channels  : 2 (stereo)')
print(f'   Duration  : {len(y_proc[0])/sr:.2f} s')
print(f'   Peak      : {20*np.log10(np.max(np.abs(y_proc))+1e-10):.1f} dBFS')
print()
print('⬇️  Downloading...')
files.download(out_name)

# Also download the analysis chart
files.download('beatlab_ai_analysis.png')
files.download('beatlab_before_after.png')

---
## 📖 Metric Reference

| Metric | What it measures | AI signature | Human signature |
|---|---|---|---|
| **Dynamic Range** | Crest factor (peak/RMS) | < 8 dB — over-limited | 12–20 dB |
| **Spectral Flatness** | How uniform frequency energy is | > 0.25 (too flat) or < 0.03 (over-processed) | 0.03–0.20 |
| **Timing Regularity** | Std-dev of inter-beat intervals | < 3 ms jitter (machine-perfect) | 5–25 ms natural variance |
| **Stereo Width** | Side/Mid energy ratio | > 0.75 (too wide) or < 0.03 (near-mono) | 0.15–0.60 |
| **HF Energy** | Energy above 15 kHz as % of total | < 0.002% (dead HF) or > 7% (boosted) | 0.2–3% |
| **Noise Floor** | Level of quietest 5% of frames | Below −90 dBFS (suspiciously clean) | −75 to −60 dBFS |

### Scoring
- **0–38%** → Likely human-produced
- **38–65%** → Uncertain / ambiguous (may be heavily processed human audio)
- **65–100%** → Likely AI-generated

### Important note
Heavily mastered, quantised, or programmed human beats (e.g. electronic music with tight drum programming) can score higher than expected. The score is a *signal*, not a verdict.
